# 예선 — EDA 및 피처 엔지니어링

HD현대 AI Challenge 2023 예선 단계 데이터셋(정형 표 형식, 단일 train/test) 으로부터 **선박 입항 후 대기시간(`CI_HOUR`)** 회귀 예측을 위한 피처를 만듭니다.

후속 노트북(`02_xgboost_catboost_ensemble.ipynb`) 에서 동일 결과를 재사용할 수 있도록 처리된 학습/테스트 셋을 `data/processed/` 에 저장합니다.


## Imports

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

DATA_DIR = Path("data")
PROCESSED_DIR = DATA_DIR / "processed" / "preliminaries"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 데이터 로드

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv").drop(columns=["SAMPLE_ID"])
test = pd.read_csv(DATA_DIR / "test.csv").drop(columns=["SAMPLE_ID"])
print(f"train: {train.shape}, test: {test.shape}")
train.head()

## 타겟 분포 확인

In [ ]:
TARGET = "CI_HOUR"
print(train[TARGET].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(train[TARGET], ax=axes[0], bins=80, color="steelblue")
axes[0].set_title("CI_HOUR 분포")
axes[0].set_yscale("log")
sns.histplot(np.log1p(train[TARGET]), ax=axes[1], bins=80, color="steelblue")
axes[1].set_title("log1p(CI_HOUR) 분포")
plt.tight_layout()
plt.show()

`CI_HOUR` 는 0 근처에 강하게 치우쳐 있습니다. 이후 모델은 MAE 기반으로 평가하며(대회 평가지표), 분포 형태로 보아 부스팅 계열 + 양수 클리핑이 적합합니다.

## 결측 패턴 확인

In [ ]:
missing = train.isna().sum()
missing[missing > 0].sort_values(ascending=False)

`BREADTH` 가 결측인 행은 선박 메타정보 자체가 비는 케이스라 학습에서 제외합니다. 기상 변수(`U_WIND`, `V_WIND`, `AIR_TEMPERATURE`, `BN`) 는 일부 (0,0,NaN,0) 패턴이 진짜 결측을 가린 케이스이므로 NaN 으로 통일합니다.

In [ ]:
def clean_basics(frame: pd.DataFrame, has_target: bool) -> pd.DataFrame:
    out = frame.copy()
    if has_target:
        out = out[out["BREADTH"].notnull()].reset_index(drop=True)
    mask = (
        (out["U_WIND"] == 0)
        & (out["V_WIND"] == 0)
        & (out["AIR_TEMPERATURE"].isna())
        & (out["BN"] == 0)
    )
    out.loc[mask, ["U_WIND", "V_WIND", "AIR_TEMPERATURE", "BN"]] = np.nan
    return out


train = clean_basics(train, has_target=True)
test = clean_basics(test, has_target=False)

## 시간 파생 피처

In [ ]:
def add_calendar_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out["ATA"] = pd.to_datetime(out["ATA"])
    out["year"] = out["ATA"].dt.year
    out["month"] = out["ATA"].dt.month
    out["day"] = out["ATA"].dt.day
    out["hour"] = out["ATA"].dt.hour
    out["minute"] = out["ATA"].dt.minute
    out["weekday"] = out["ATA"].dt.weekday
    out["quarter"] = (out["month"] - 1) // 3 + 1
    return out


train = add_calendar_features(train)
test = add_calendar_features(test)

## weather_mean 파생 피처

같은 (국가, 항구, 기상 조건) 조합에서 도착 시각(`ATA`) 대비 출항 추정 시각(`BTA = ATA + CI_HOUR`) 의 평균을 학습 셋에서 구하고, 테스트의 동일 조합에 매핑합니다. **타깃 누설 방지를 위해 학습 셋에서만 집계**한 뒤 매핑합니다.


In [ ]:
WEATHER_KEYS = ["ARI_CO", "U_WIND", "V_WIND", "AIR_TEMPERATURE", "BN"]


def build_weather_mean(train_frame: pd.DataFrame) -> pd.DataFrame:
    work = train_frame.copy()
    work["BTA"] = work["ATA"] + pd.to_timedelta(work["CI_HOUR"], "h")
    pivot = (
        work.groupby(WEATHER_KEYS)["BTA"]
        .mean()
        .reset_index()
        .rename(columns={"BTA": "BTA_mean"})
    )
    return pivot


def attach_weather_mean(frame: pd.DataFrame, lookup: pd.DataFrame) -> pd.DataFrame:
    merged = frame.merge(lookup, how="left", on=WEATHER_KEYS)
    merged["weather_mean"] = (merged["BTA_mean"] - merged["ATA"]).dt.total_seconds() / 3600
    merged["weather_mean"] = merged["weather_mean"].clip(lower=0)
    return merged.drop(columns=["BTA_mean"])


lookup = build_weather_mean(train)
train = attach_weather_mean(train, lookup)
test = attach_weather_mean(test, lookup)
print(f"weather_mean coverage — train: {train['weather_mean'].notna().mean():.2%}, test: {test['weather_mean'].notna().mean():.2%}")

## 카테고리 인코딩

테스트에서 처음 등장하는 라벨도 안전하게 처리하도록 `LabelEncoder.classes_` 를 확장합니다.


In [ ]:
CATEGORICAL_FEATURES = [
    "FLAG", "ARI_CO", "ARI_PO", "SHIP_TYPE_CATEGORY", "SHIPMANAGER",
    "year", "month", "day", "hour", "minute", "weekday",
]


def encode_categoricals(train_frame: pd.DataFrame, test_frame: pd.DataFrame, columns: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_out, test_out = train_frame.copy(), test_frame.copy()
    for col in columns:
        encoder = LabelEncoder().fit(train_out[col].astype(str))
        train_out[col] = encoder.transform(train_out[col].astype(str))
        new_labels = set(test_out[col].astype(str)) - set(encoder.classes_)
        if new_labels:
            encoder.classes_ = np.append(encoder.classes_, list(new_labels))
        test_out[col] = encoder.transform(test_out[col].astype(str))
    return train_out, test_out


train, test = encode_categoricals(train, test, CATEGORICAL_FEATURES)

## Target encoding (CatBoost 파이프라인용)

In [ ]:
def add_target_encodings(train_frame: pd.DataFrame, test_frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_out, test_out = train_frame.copy(), test_frame.copy()

    groupings = [
        (["ARI_CO", "ARI_PO"], "CO_PO_mean"),
        (["ARI_PO", "ARI_CO", "month", "year"], "CO_PO_ym_mean"),
        (["ARI_PO", "ARI_CO", "SHIP_TYPE_CATEGORY", "year"], "CO_PO_SH_y_mean"),
    ]
    for keys, name in groupings:
        agg = train_out.groupby(keys)["CI_HOUR"].mean().reset_index().rename(columns={"CI_HOUR": name})
        train_out = train_out.merge(agg, how="left", on=keys)
        test_out = test_out.merge(agg, how="left", on=keys)
        global_mean = train_out[name].mean()
        test_out[name] = test_out[name].fillna(global_mean)
    return train_out, test_out


train, test = add_target_encodings(train, test)

## 최종 컬럼 정리 및 저장

In [ ]:
TIME_DROP = ["ATA"]
train = train.drop(columns=TIME_DROP)
test = test.drop(columns=TIME_DROP)

train.to_parquet(PROCESSED_DIR / "train.parquet")
test.to_parquet(PROCESSED_DIR / "test.parquet")
print(f"saved processed train/test to {PROCESSED_DIR}")
train.head()

## 정리

- `BREADTH` 결측 학습 행 제거, 기상 변수의 위장 결측을 NaN 으로 통일.
- `ATA` 기반 시간 파생 피처(year/month/day/hour/minute/weekday/quarter) 생성.
- 학습 셋에서만 집계한 `weather_mean` 파생 피처를 양 셋에 매핑(누설 방지).
- 카테고리 인코딩 + 세 가지 target encoding(CO/PO, CO/PO/y/m, CO/PO/SHIP/year).
- 결과는 parquet 으로 저장해 다음 노트북이 동일 입력으로 모델만 비교할 수 있게 했습니다.
